In [1]:
import pandas as pd, os, datetime
import numpy as np

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
gen_details = pd.read_csv(f"{nmap_path}/gen_details.csv")
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")
hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])

For this example, I'll focus on plotting a 12 day period from 2018/11/23 to 2018/12/12.
No generators are in heatwave conditions from 11/09 to 11/22

In [4]:
# sdate, edate = '2018-11-09','2018-12-25'
sdate, edate = '2018-10-06','2019-04-26'

Separating the states

In [6]:
def process_group(grp, gen_fpath, hw_tseries, start_date=sdate, end_date=edate):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    gen_locs = gen_fpath + '/' + grp['DUID'] + ".csv"
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]

    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        dfs.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )

    return merged

In [7]:
def process_all(gen_details, gen_fpath, hw_tseries, start_date=sdate, end_date=edate):
    """
    Process all generator data in gen_details without grouping. Ignores loads and batteries.
    
    Parameters:
        gen_details (pd.DataFrame): DataFrame containing at least 'duid' column.
        gen_fpath (str): Directory path containing CSV files named by DUID.
        hw_tseries (pd.DataFrame): Heatwave timeseries data with 'time' and 'DUID'.
        start_date (str): Start date for filtering time series.
        end_date (str): End date for filtering time series.
    
    Returns:
        pd.DataFrame: Merged dataframe with summed TOTALMWh per DUID per hour and heatwave info.
    """
    # Build file paths for all DUIDs,ignoring batteries

    gen_locs = gen_fpath + '/' + gen_details['DUID'] + ".csv"

    # Read all existing CSVs
    dfs = [pd.read_csv(fp) for fp in gen_locs if os.path.exists(fp)]
    
    if not dfs:
        print("[WARN] No data files found for any DUID.")
        return pd.DataFrame()  # Return empty if no files
    
    # Concatenate all data
    dfs = pd.concat(dfs, ignore_index=True)

    # This is in case of accidental mid-file headers
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)

    # Convert time to datetime and filter
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    # Group by DUID and hourly time, sum TOTALMWh
    agg_func = {'TOTALMWh':'sum','TOTALCLEARED':'sum','AGCSTATUS':'min'}
    grouped = dfs.groupby(['DUID', pd.Grouper(freq='1h')]).agg(agg_func).reset_index()

    # Sort hw_tseries and convert column types
    hw_sorted = hw_tseries.sort_values(by=['time', 'DUID'])
    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    # Merge with heatwave timeseries
    merged = pd.merge_asof(
        grouped.sort_values(by=['time', 'DUID']),
        hw_sorted,
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )
    
    return merged

In [8]:
df = process_all(gen_details, gen_fpath, hw_tseries)
df = df.merge(gen_details[['DUID', 'region','fuel_source_primary']], left_on='DUID', right_on='DUID', how='left')

Using KSP1 as the first generator in the heatwave for QLD and SUNRSF1 as first for VIC, STWF1 (NSW) has next longest hw of 5 days

In [9]:
def highLights(df, fig, variable, level, mode, fillcolor, layer):
    """
    Set a specified color as background for given
    levels of a specified variable using a shape.
    
    Keyword arguments:
    ==================
    fig -- plotly figure
    variable -- column name in a pandas dataframe
    level -- int or float
    mode -- set threshold above or below
    fillcolor -- any color type that plotly can handle
    layer -- position of shape in plotly fiugre, like "below"
    
    """
    
    if mode == 'above':
        m = df[variable].gt(level)
    
    if mode == 'below':
        m = df[variable].lt(level)
        
    df1 = df[m].groupby((~m).cumsum())['time'].agg(['first','last'])

    for index, row in df1.iterrows():
        #print(row['first'], row['last'])
        fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0=row['first'],
            y0=0,
            x1=row['last'],
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(100,100,100,0.2)",
            layer=layer
        )
    return(fig)


In [10]:
def plot_agg_group(
    grouped_df, group_col, title='Time Series Plot', y='TOTALMWh',
    highlight=False, highlight_mode='union'
):
    fig = go.Figure()

    for group_name, group in grouped_df.groupby(group_col):
        fig.add_trace(go.Scatter(
            x=group['time'],
            y=group[y],
            mode='lines',
            name=str(group_name)
        ))

    if highlight:
        if highlight_mode == 'union':
            # Highlight where any group has EHF_flag==1 (union)
            highlight_times = (
                grouped_df.groupby('time')['EHF_flag']
                .max()
                .reset_index()
            )
            fig = highLights(
                df=highlight_times,
                fig=fig,
                variable='EHF_flag',
                level=0,
                mode='above',
                fillcolor='rgba(255,0,0,0.1)',
                layer='below'
            )
        elif highlight_mode == 'per_group':
            # Highlight per group
            for group_name, group in grouped_df.groupby(group_col):
                fig = highLights(
                    df=group,
                    fig=fig,
                    variable='EHF_flag',
                    level=0,
                    mode='above',
                    fillcolor='rgba(255,0,0,0.1)',
                    layer='below'
                )
        else:
            raise ValueError("highlight_mode must be 'union' or 'per_group'")

    fig.update_layout(
        title=title,
        xaxis_title='Time',
        yaxis_title='Total MWh',
        template='plotly_white',
        legend_title=group_col
    )

    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(count=1,
                         label="1m",
                         step="month",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )

    return fig

In [17]:
def plot_multivars(df, title='Time Series Plot',lines=['TOTALMWh'], highlight = False):
    fig = go.Figure()

    for line in lines:
        fig.add_trace(go.Scatter(
            x=df['time'],
            y=df[line],
            mode='lines',
            name=str(line)
        ))
        
        if highlight == True:
            # Highlight the EHF flag
            fig = highLights(
                df=df,             # Only this group's data
                fig=fig,
                variable='EHF_flag',  # Column to check
                level=0,              # Threshold
                mode='above',         # or 'below'
                fillcolor='rgba(255,0,0,0.1)',  # Semi-transparent red
                layer='below'
            )

    fig.update_layout(
        title=title,
        xaxis_title='Time',
        yaxis_title='Total MWh',
        template='plotly_white',
        legend_title='Legend'
    )

    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=1,
                         label="1d",
                         step="day",
                         stepmode="backward"),
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )

    return fig
    

In [18]:
def plot_unit(df, title):
    fig = go.Figure()
    
    fig.add_trace(
        go.Scatter(x=df['time'], y=df['TOTALMWh']))
    
    # Set title
    fig.update_layout(
        title_text=title
    )
    
    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=1,
                         label="1d",
                         step="day",
                         stepmode="backward"),
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )
    
    fig = highLights(df=df, fig = fig, variable = 'EHF_flag', level = 0, mode = 'above',
                   fillcolor = 'rgba(200,0,200,0.2)', layer = 'below')
    
    return fig


In [19]:
test = df[df['DUID']== 'BOCORWF1']
test2 = df[df['DUID'] == 'STWF1']

fig1 = plot_multivars(test, title="Hourly energy generation at Boco Rock (QLD)", lines=['TOTALMWh','EHF_val'],highlight=True)
fig2 = plot_multivars(test2, title='Hourly energy generation at Silverton Wind Farm (NSW)', lines=['TOTALMWh','EHF_val'],highlight=True)
fig1.show()
fig2.show()
fig1.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/boco_rock.html')
fig2.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/silverton.html')

In [ ]:
agg_func = {'TOTALMWh':'sum','EHF_flag':'max', 'EHF_val':'max','HW_event_day':'first'}
reg_grp = df.groupby(['region', 'time']).aggregate(agg_func).reset_index()
fuel_grp = df.groupby(['fuel_source_primary', 'time']).aggregate(agg_func).reset_index()

reg_grp['normalised'] = reg_grp.groupby('region')['TOTALMWh'].transform(
                        lambda x: (x - x.min()) / (x.max() - x.min()))
fuel_grp['normalised'] = fuel_grp.groupby('fuel_source_primary')['TOTALMWh'].transform(
                        lambda x: (x - x.min()) / (x.max() - x.min()))

fig1 = plot_agg_group(reg_grp, group_col='region', title='Hourly energy generation (MWh) by region',highlight=True, highlight_mode='per_group')
fig1.show()
fig2 = plot_agg_group(fuel_grp, group_col='fuel_source_primary', title='Normalised hourly energy generation by technology',highlight=True,y='normalised')
fig2.show()

fig1.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/regional_gen.html')
fig2.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/fueltype_gen.html')

In [ ]:
demand = pd.read_csv('/scratch/ng72/ms5578/time_series/state_demand.csv')
demand['time'] = pd.to_datetime(demand['time'])
demand = demand.set_index('time').sort_index()
demand = demand.loc[sdate:edate]

In [ ]:
state_df = df.groupby('region')    
states = {name: group for name, group in state_df}
qld_df,nsw_df,vic_df,sa_df,tas_df = states['QLD1'],states['NSW1'],states['VIC1'],states['SA1'],states['TAS1']

In [ ]:
vic_df = nsw_df.groupby(['fuel_source_primary', 'time']).aggregate(agg_func).reset_index()
vic_df['normalised'] = vic_df.groupby('fuel_source_primary')['TOTALMWh'].transform(
                        lambda x: (x - x.min()) / (x.max() - x.min()))

vic_demand = demand[demand["REGIONID"] ==  'NSW1']

fig = plot_agg_group(vic_df, group_col='fuel_source_primary',
                    title='Energy Generation in NSW (MWh)',
                    highlight_mode='union',
                    highlight=True,
                    y='TOTALMWh')

# fig.add_trace(go.Scatter(
#             x=vic_demand.index,
#             y=vic_demand['TOTALDEMAND'],
#             mode='lines',
#             line=dict(
#                 color='blue',
#                 width=1,
#                 dash='dot'
#                 ),
#             name=str('State Demand (MWh)')
#         ))

fig.show()
# fig.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/vic_gen.html')

In [ ]:
tas_df = tas_df.groupby(['fuel_source_primary', 'time']).aggregate(agg_func).reset_index()
tas_df['normalised'] = tas_df.groupby('fuel_source_primary')['TOTALMWh'].transform(
                        lambda x: (x - x.min()) / (x.max() - x.min()))

tas_demand = demand[demand["REGIONID"] ==  'TAS1']
tot_demand = demand.groupby(['time']).aggregate('sum').reset_index()

fig = plot_agg_group(tas_df, group_col='fuel_source_primary',
                    title='Energy Generation in Tasmania (MWh)',
                    highlight_mode='union',
                    highlight=True,
                    y='TOTALMWh')

fig.add_trace(go.Scatter(
            x=tas_demand.index,
            y=tas_demand['TOTALDEMAND'],
            mode='lines',
            line=dict(
                color='blue',
                width=1,
                dash='dot'
                ),
            name=str('State Demand (MWh)')
        ))

# fig.add_trace(go.Scatter(
#             x=tot_demand['time'],
#             y=tot_demand['TOTALDEMAND'],
#             mode='lines',
#             line=dict(
#                 color='black',
#                 width=1,
#                 dash='dot'
#                 ),
#             name=str('Total Demand (MWh)')
#         ))

fig.show()
fig.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/tas_gen.html')

In [ ]:
event_df = df[df['HW_event_day'].notna()].copy()
event_df['time_of_day'] = pd.to_datetime(event_df['time'].dt.strftime("1900-01-01 %H:%M:%S"))

# Filter for Solar
solar_df = event_df[event_df['fuel_source_primary'] == "Wind"]
solar_df['normalised'] = solar_df.groupby('DUID')['TOTALMWh'].transform(
                        lambda x: (x - x.min()) / (x.max() - x.min()))
solar_df.HW_event_day = solar_df.HW_event_day.astype(int)


# Group by HW_event_day and time_of_day, then average
avg_df = (
    solar_df.groupby(['HW_event_day', 'time_of_day'])['TOTALMWh']
    .mean()
    .reset_index()
)

rgb = px.colors.convert_colors_to_same_type(px.colors.sequential.Magma)[0]

colorscale = []
n_steps = 5  # Control the number of colors in the final colorscale
for i in range(len(rgb) - 1):
    for step in np.linspace(0, 1, n_steps):
        colorscale.append(px.colors.find_intermediate_color(rgb[i], rgb[i + 1], step, colortype='rgb'))

# Plot
fig = px.line(
    avg_df,
    x='time_of_day',
    y='TOTALMWh',
    color='HW_event_day',
    color_discrete_sequence=colorscale,
    markers=True,
    title='Average wind output during heatwave events',
    labels={'TOTALMWh': 'Average MWh', 'time_of_day': 'Time of Day'}
)

fig.update_layout(
    xaxis=dict(tickformat="%H:%M"),
    xaxis_title='Time of Day',
    yaxis_title='Average MWh'
)

fig.show()
fig.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/wind_hw_day_gen.html')

In [ ]:
agg_df = demand.reset_index().groupby('time', as_index=False).agg({'TOTALDEMAND': 'sum'})

total_row = pd.DataFrame({
    'REGIONID': ['ALL'],
    'TOTALDEMAND': [agg_df['TOTALDEMAND'].sum()]
})

demand = pd.concat([demand, total_row], ignore_index=True)

In [ ]:
hw_freq = hw_tseries.groupby('DUID')['EHF_flag'].apply(lambda x: (x == 1).sum()).reset_index(name='days_in_HW')
hw_freq = pd.merge(hw_freq,gen_details[['DUID','lat','lon']],left_on='DUID',right_on='DUID')

In [ ]:
def plot_gens_plotly(nmap,y,title,legtitle):
    fig = go.Figure()

    # Plot all generators with colorbar
    fig.add_trace(go.Scattergeo(
        lon=nmap['lon'],
        lat=nmap['lat'],
        mode='markers',
        marker=dict(
            color=nmap[y],  # Continuous color scale
            colorscale='Viridis',      # You can use 'Plasma', 'Jet', 'Cividis', etc.
            colorbar=dict(
                title=legtitle,
                thickness=15,
                len=0.75
            ),
            size=6,
            symbol='circle',
            line=dict(width=0)
        ),
        name='All Generators',
        hovertemplate=(
            "DUID: %{customdata[0]}<br>"
            "Days in HW: %{marker.color}<br>"
            "Lat: %{lat}<br>"
            "Lon: %{lon}<extra></extra>"
        ),
        customdata=nmap[['DUID']]  # Optional: shows DUID in hover
    ))

    # Configure the map
    fig.update_layout(
        title=title,
        geo=dict(
            projection_type="natural earth",
            lonaxis=dict(range=[110, 155]),
            lataxis=dict(range=[-45, -10]),
            showland=True,
            landcolor="rgb(240, 240, 240)",
            coastlinecolor="black",
            showcountries=True,
        )
    )

    return fig

In [ ]:
fig = plot_gens_plotly(hw_freq,'days_in_HW','Days spent in heatwave conditions','Days in heatwave')
fig.show()
fig.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/hw_days_per_gen.html')

In [ ]:
peak = hw_tseries.groupby('DUID')['HW_EHF_peak'].apply('max').reset_index(name='peak_EHF')
peak = pd.merge(peak,gen_details[['DUID','lat','lon']],left_on='DUID',right_on='DUID')

In [ ]:
fig = plot_gens_plotly(peak,'peak_EHF','Peak Excess Heat Factor by generator location','EHF')
fig.show()
fig.write_html('/g/data/ng72/ms5578/ID_HW_BARRA/data/output/plots/peakEHF.html')